1. Ingest ----> 2. prepare ----->  3. store -----> 4. retreivement -------> 5. augment -------> . generation

In [1]:
# step 0 : Importing the tools 
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()

api_key=os.getenv("api_key")

client=Groq(api_key=api_key)

In [2]:
# for loading the documents
from langchain_community.document_loaders import PyPDFDirectoryLoader
# for chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
# encoding of the chunks
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
# for storing the chunks 
from langchain_community.vectorstores import Chroma

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8732\2073991335.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## prepare
### Chunking

In [3]:
# Loading the pdf
pdfs_loader=PyPDFDirectoryLoader("tesla-annual-reports")
#Chunking
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=512,
    chunk_overlap=16
)

tesla_chunks=pdfs_loader.load_and_split(text_splitter)


## Store

In [4]:
tesla_collection="tesla-10k-2019-to-2023"

embedding_model=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
# embedding and storing the chunks 
vectorstore = Chroma.from_documents(
    tesla_chunks,
    embedding_model,
    collection_name=tesla_collection,
    persist_directory="./tesla_db"
)
# storing the chunks permanently
vectorstore.persist()
# Connect to the existing vector database on disk 
# This lets us skip re-reading the PDFs and re-generating embeddings
vectorstore_persisted=Chroma(
    collection_name=tesla_collection,
    persist_directory="./tesla_db",
    embedding_function=embedding_model
)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8732\2141288676.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4250.13it/s]
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8732\2141288676.py:12: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8732\2141288676.py:15: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version

## Retreivement

In [5]:
query = "What is the annual revenue in the year 2022 ?"

docs=vectorstore_persisted.similarity_search(query,k=5)

for i,doc in enumerate(docs):
    print(f"Retrieved chunk {i+1} : \n")
    print(doc.page_content.replace("\t"," "))
    print("\n")

Retrieved chunk 1 : 

Our cash flows provided by operating activities in 2023 and 2022 were $13.26 billion and $14.72 billion, respectively, representing a decrease of $1.47
billion. Capital expenditures amounted to $8.90 billion in 2023, compared to $7.16 billion in 2022, representing an increase of $1.74 billion. Sustained
growth has allowed our business to generally fund itself, and we will continue investing in a number of capital-intensive projects and research and
development in upcoming periods.
33


Retrieved chunk 2 : 

(100)%
$
203 
Not meaningful
During the year ended December 31, 2022, we recorded an impairment loss of $204 million as well as realized gains of $64 million in connection
with converting our holdings of digital assets into fiat currency. We also recorded other expenses of $36 million during the second quarter of the year
ended December 31, 2022, related to employee terminations.
Interest Income
Year Ended December 31,
2023 vs. 2022 Change
2022 vs. 2021 Change


## Generate  : 

In [6]:
model_name='llama-3.3-70b-versatile'

qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer user questions only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

.
"""


qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""


In [7]:
user_input = "What is the annual revenue in the year 2022 ?"

retriever=vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}

)

relevant_document_chunks = retriever.invoke(user_input)

# Composing the response
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

prompt=[
    {"role":"system","content":(qna_system_message)},
    {"role":"user","content":(qna_user_message_template.format(
        context=context_for_query,
        question=user_input
    ))}
]

try:
    response=client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )
    prediction=response.choices[0].message.content.strip()

except Exception as e :
    prediction = f"Sorry , I encountered the following error : \n {e}"

print(prediction)

The annual revenue for the year 2022 is not explicitly stated in the provided context. However, various revenue components are mentioned, such as:

- Automotive leasing revenue (decrease of $356 million from 2022 to 2023, but the actual 2022 revenue is not provided)
- Services and other revenue (increased by $2.23 billion from 2022 to 2023, but the actual 2022 revenue is not provided)
- Energy generation and storage revenue (increased by $2.13 billion from 2022 to 2023, but the actual 2022 revenue is not provided)

Without the actual revenue values for 2022, the total annual revenue for the year 2022 cannot be determined from the provided context.


In [ ]:
import gradio as gr

def rag_qa(query):
    relevant_document_chunks = retriever.invoke(query)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    prompt = [
        {'role':'system', 'content': qna_system_message},
        {'role': 'user', 'content': qna_user_message_template.format(
             context=context_for_query,
             question=query
            )
        }
    ]

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=prompt,
            temperature=0
        )

        prediction = response.choices[0].message.content.strip()
    except Exception as e:
        prediction = f'Sorry, I encountered the following error: \n {e}'

    return prediction

print("Gradio imported and rag_qa function defined.")

iface = gr.Interface(fn=rag_qa, inputs='text', outputs='text', title='Tesla Annual Report QA')
iface.launch(debug=True, share=True)

Gradio imported and rag_qa function defined.
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/06/11 13:09:36 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


Created dataset file at: .gradio\flagged\dataset1.csv
